# Classificador de Qualidade de Lances de Xadrez

**Disciplina:** Paradigmas de Aprendizagem de Máquina  

---

## Sobre este Notebook

Este é o **notebook principal** (orquestrador) do projeto. Ele apresenta os resultados de uma versão selecionada usando dados pré-computados.

- Para **re-executar o pipeline** (download, filtragem, rotulagem, treino), use os notebooks por versão: `v1_posicional.ipynb`, `v2_tatica.ipynb`, `v3_lookahead.ipynb`.
- Para **comparar as 3 versões** lado a lado, use `comparacao.ipynb`.
- Para **mudar a versão** apresentada aqui, altere `config = V3` na célula de configuração.

### Versões do pipeline

| Versão | Features | Descrição |
|--------|----------|-----------|
| **V1** | 33 posicionais | Material, mobilidade, segurança do rei, estrutura de peões, controle do centro |
| **V2** | 52 (33 + 19 táticas) | V1 + peças indefesas, capturas com ganho, cravadas, rei avançado, tensão |
| **V3** | 67 (52 + 15 look-ahead) | V2 + deltas antes/depois, resposta do adversário, Static Exchange Evaluation (SEE) |

### Abordagem resumida

| Etapa | Método |
|-------|--------|
| Dados | Partidas reais do Lichess (CC0), filtradas por rating e tempo |
| Rotulagem | Avaliação posicional via Stockfish (depth 15): bom (≥ −50 cp), ruim (≤ −150 cp) |
| Features | 33 → 52 → 67 features extraídas com `python-chess` |
| Modelos | Decision Tree + Random Forest, com `class_weight="balanced"` |
| Métricas | Foco em recall e F1 da classe "ruim" — detectar erros é o objetivo |

In [ ]:
import json
import os
import random
import sys
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*sklearn.utils.parallel.delayed.*", category=UserWarning)
plt.rcParams.update({"font.size": 11, "figure.facecolor": "white"})
sns.set_style("whitegrid")

PROJECT_ROOT = Path("..").resolve()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

DATA_DIR = Path("data")

from version_config import V1, V2, V3, ALL_VERSIONS
from notebook_utils import *

print(f"Diretório de trabalho: {os.getcwd()}")
print("Setup OK")

In [ ]:
config = V3

print(f"Versão selecionada: {config.label}")
print(f"Features CSV      : {config.features_csv}")
print(f"Modelos           : {config.models_dir}")
print(f"Avaliação         : {config.eval_dir}")

---

## 2. Coleta e Descrição dos Dados

### Fonte

Os dados vêm da **Lichess open database** ([database.lichess.org](https://database.lichess.org)), sob licença **CC0** (domínio público).

- **Ficheiro:** `lichess_db_standard_rated_2015-01.pgn.zst` (~272 MiB comprimido)

### Filtros aplicados

| Filtro | Critério | Justificativa |
|--------|----------|---------------|
| Rating | Ambos 1400–1700 (Lichess) | Faixa-alvo equivalente a 1200–1500 Chess.com |
| Tempo | Blitz/Rapid (3–10 min) | Jogos sérios mas com pressão temporal |
| Término | Normal (mate/resignação) | Evitar jogos decididos por timeout |
| Variante | Standard | Sem Chess960 etc. |
| Fase | Lances 8 a 40 | Meio-jogo |
| Amostragem | 10% das partidas válidas | Controle de volume, seed=42 |

In [ ]:
df_filtered = pd.read_csv(DATA_DIR / "filtered" / "moves_filtered.csv")
print_dataset_stats(df_filtered)
print()
df_filtered.head()

In [ ]:
plot_rating_distribution(df_filtered)

---

## 3. Rotulagem via Stockfish

$$\delta = \text{eval\_depois} - \text{eval\_antes}$$

| Classe | Condição | Significado |
|--------|----------|-------------|
| **Bom** | δ ≥ −50 cp | Perda ≤ 0.5 peão |
| **Descartado** | −150 < δ < −50 | Zona cinzenta |
| **Ruim** | δ ≤ −150 cp | Perda ≥ 1.5 peão |

In [ ]:
df_scored = pd.read_csv(DATA_DIR / "labeled" / "moves_all_scored.csv")
df_labeled = pd.read_csv(DATA_DIR / "labeled" / "moves_labeled.csv")
print_labeling_stats(df_scored, df_labeled)

In [ ]:
plot_labeling_analysis(df_scored, df_labeled)

---

## 4. Engenharia de Features

In [ ]:
df_features = pd.read_csv(config.features_csv)
feature_cols = [c for c in df_features.columns if c != "label"]
print_features_stats(df_features, config)
print(f"\nEstatísticas descritivas:")
df_features[feature_cols].describe().round(2)

In [ ]:
plot_correlation_matrix(df_features, feature_cols)

---

## 5. Treino dos Modelos

Split **70/15/15** (treino/validação/teste) com estratificação. `class_weight="balanced"`. `GridSearchCV` com 5 folds, métrica F1 da classe "ruim".

In [ ]:
X = df_features.drop(columns=["label"])
y = (df_features["label"] == "ruim").astype(int)

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=RANDOM_SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, stratify=y_temp, random_state=RANDOM_SEED
)

print_split_info(X_train, y_train, X_val, y_val, X_test, y_test)

In [ ]:
dt, rf, feature_names = config.load_models()
print_model_params(dt, rf, config)

---

## 6. Avaliação

In [ ]:
print_classification_reports(dt, rf, X_test, y_test)

In [ ]:
df_results = build_results_table(dt, rf, X_test, y_test)
print("Tabela comparativa — Conjunto de teste:\n")
df_results

In [ ]:
plot_confusion_matrices(dt, rf, X_test, y_test)

In [ ]:
plot_feature_importances_side_by_side(dt, rf, feature_names)

In [ ]:
plot_roc_pr_curves(dt, rf, X_test, y_test)

In [ ]:
plot_metrics_comparison(dt, rf, X_test, y_test)

In [ ]:
plot_learning_curves(dt, rf, X_train, y_train)

---

## 7. Comparação com Versões Anteriores

In [ ]:
prior_configs = config.prior_versions
if prior_configs:
    all_configs = prior_configs + [config]
    plot_version_metrics_bars(all_configs, X_test, y_test)
    plot_version_roc_pr_overlay(all_configs, X_test, y_test)
else:
    print("V1 não tem versões anteriores para comparar.")

---

## 8. Diagnóstico

In [ ]:
plot_diagnostic(df_features, feature_cols, f"V{config.version}")

---

## 9. Interpretação

### Regras da Árvore de Decisão

In [ ]:
plot_decision_tree(dt, feature_names)

In [ ]:
print_tree_rules(config)

### Análise de Erros

In [ ]:
show_error_examples(config)

---

## 10. Conclusões

| Métrica (RF) | V1 | V2 | V3 |
|-------------|-----|-----|-----|
| **Accuracy** | 0.6827 | 0.6912 | **0.7849** |
| **F1 (ruim)** | 0.3525 | 0.3664 | **0.4312** |
| **ROC-AUC** | 0.6837 | 0.7079 | **0.7678** |

O ciclo **treino → diagnóstico → melhoria → re-treino** é o resultado central: cada iteração foi motivada por evidência quantitativa e resultou em melhoria mensurável.

Para a análise completa por versão, consulte:
- `v1_posicional.ipynb` — V1 + diagnóstico
- `v2_tatica.ipynb` — V2 + diagnóstico
- `v3_lookahead.ipynb` — V3 + diagnóstico
- `comparacao.ipynb` — comparação V1 vs V2 vs V3